# PCA(주성분 분석) 완전 정복 실습

## 이 노트북에서 배우는 것
- PCA가 **무엇을** 하는지 직관적으로 이해한다
- PCA가 **왜** 필요한지 알게 된다
- PCA가 **어떻게** 작동하는지 수학적으로 이해하고 NumPy로 직접 구현한다
- scikit-learn으로 실제 데이터에 적용하고 결과를 해석한다

---

## 1분 요약: PCA란?

> **PCA는 '데이터를 가장 잘 설명하는 새로운 축'을 찾아주는 방법이다.**

예를 들어 키와 몸무게 데이터가 있다고 하자. 두 값은 서로 강하게 연관되어 있다 — 키가 크면 보통 몸무게도 크다. 이때 '키'와 '몸무게' 두 축 대신, **'체격'이라는 새로운 하나의 축**으로도 거의 대부분의 정보를 표현할 수 있다. PCA는 바로 이런 '체격' 같은 새로운 축을 자동으로 찾아준다.

### 왜 필요한가?
1. **시각화**: 100차원 데이터는 그릴 수 없지만, PCA로 2~3차원으로 줄이면 그래프로 볼 수 있다
2. **노이즈 제거**: 중요한 방향만 남기고 작은 변동은 버린다
3. **계산 속도**: 특성 수가 줄면 학습이 빨라진다
4. **다중공선성 해소**: 서로 상관된 특성들을 독립된 축으로 바꾼다

## 0. 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 한글 폰트 (Colab)
!apt-get install -qq fonts-nanum > /dev/null
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
np.random.seed(42)
print('준비 완료')

## 1. 직관: 분산이 큰 방향 = 정보가 많은 방향

### 핵심 아이디어
PCA의 핵심 통찰은 이것이다:

> **"데이터가 많이 퍼져 있는 방향이 가장 중요한 방향이다."**

왜 그럴까? 한번 생각해보자. 만약 어떤 방향으로 데이터가 거의 퍼져있지 않다면 (= 분산이 0에 가깝다면), 그 방향의 좌표는 모든 데이터가 거의 같은 값을 갖는다. 즉, **그 축은 데이터를 구분하는 데 쓸모가 없다.** 반대로 데이터가 크게 퍼져 있는 방향은 데이터 포인트들을 잘 구분해주는 '유용한 축'이다.

### 2D 예제로 눈으로 보기
상관관계가 있는 2차원 데이터를 만들어보자. 이 데이터는 대각선 방향으로 길쭉하게 퍼져 있을 것이다. 그 '길쭉한 방향'이 바로 **제1 주성분(PC1)** 이다.

In [ ]:
mean = [0, 0]
cov = [[3, 2], [2, 2]]  # 두 변수에 양의 상관관계
X = np.random.multivariate_normal(mean, cov, 300)

plt.scatter(X[:, 0], X[:, 1], alpha=0.5)
plt.axis('equal')
plt.title('원본 2D 데이터 — 대각선 방향으로 길쭉하다')
plt.xlabel('x1'); plt.ylabel('x2')
plt.show()

### 무엇이 보이는가?
- 데이터가 **오른쪽 위 ↔ 왼쪽 아래** 방향으로 길쭉하게 퍼져 있다
- 이 방향이 바로 분산이 가장 큰 방향, 즉 **PC1**이 될 것이다
- 그와 **수직**인 방향(왼쪽 위 ↔ 오른쪽 아래)은 분산이 훨씬 작다 → **PC2**

> 💡 **PCA의 두 가지 중요한 성질**
> 1. 주성분들은 서로 **직교(수직)** 한다 → 독립적인 정보
> 2. 주성분들은 **분산이 큰 순서**대로 정렬된다 → PC1이 가장 중요, PC2는 그다음, ...

## 2. PCA의 수학: 5단계 레시피

PCA는 겁나는 수학처럼 보이지만, 실제로는 5단계의 간단한 과정이다.

### 단계 1: 중심화 (Centering)
각 특성에서 평균을 빼서 데이터의 중심을 원점(0)으로 옮긴다.
$$X_c = X - \bar{X}$$
왜? 공분산을 계산할 때 평균이 0이면 수식이 단순해지고, 주성분이 '원점을 지나는 직선'으로 정의되기 때문이다.

### 단계 2: 공분산 행렬 계산
$$C = \frac{1}{n-1} X_c^T X_c$$
공분산 행렬은 **'각 특성들이 서로 어떻게 함께 변하는가'** 를 담은 표다.
- 대각선: 각 특성의 분산(자기 자신과의 변동)
- 대각선 바깥: 특성 i와 특성 j가 함께 증감하는 정도

### 단계 3: 고유값 분해 (Eigendecomposition)
$$C \mathbf{v} = \lambda \mathbf{v}$$
공분산 행렬 $C$의 **고유벡터(eigenvector) $\mathbf{v}$** 와 **고유값(eigenvalue) $\lambda$** 를 찾는다.

**직관적 의미:**
- 고유벡터 = 주성분의 **방향**
- 고유값 = 그 방향으로의 **분산 크기** (= 그 축이 얼마나 중요한가)

### 단계 4: 정렬
고유값이 **큰 순서**로 고유벡터를 정렬한다. 첫 번째가 PC1, 두 번째가 PC2, ...

### 단계 5: 투영(Projection)
상위 $k$개의 고유벡터로 만든 행렬 $V_k$에 데이터를 곱해서 새 좌표를 얻는다.
$$Z = X_c \cdot V_k$$
$Z$가 바로 **차원 축소된 데이터**다.

---
이제 이 5단계를 NumPy로 구현해보자.

In [ ]:
def pca_numpy(X, n_components):
    # 단계 1: 중심화
    mu = X.mean(axis=0)
    X_c = X - mu

    # 단계 2: 공분산 행렬
    C = np.cov(X_c, rowvar=False)

    # 단계 3: 고유값 분해
    # eigh는 대칭행렬 전용 — 공분산 행렬은 항상 대칭이므로 적합
    # 반환값은 오름차순이므로 뒤집어야 함
    eigvals, eigvecs = np.linalg.eigh(C)

    # 단계 4: 내림차순 정렬 (큰 고유값이 앞)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # 단계 5: 상위 k개 주성분으로 투영
    components = eigvecs[:, :n_components]
    Z = X_c @ components

    # 설명 분산 비율 = 각 고유값 / 고유값 총합
    explained_var_ratio = eigvals[:n_components] / eigvals.sum()
    return Z, components, eigvals, explained_var_ratio, mu

Z, comps, eigvals, evr, mu = pca_numpy(X, n_components=2)
print('고유값 (= 각 축의 분산):', eigvals)
print('설명 분산 비율 (= 각 축이 전체의 몇 %를 설명):', evr)
print('주성분 벡터 (열 방향):\n', comps)

### 결과 해석
- **첫 번째 고유값**이 두 번째보다 훨씬 크다 → PC1이 데이터의 대부분의 분산을 설명한다
- **설명 분산 비율**의 첫 값이 약 0.8 이상이라면 → 2D를 1D로 줄여도 80% 이상의 정보를 보존할 수 있다는 뜻!
- **주성분 벡터**는 방향을 나타내는 단위벡터다. 각 열이 하나의 주성분

이제 찾은 주성분 축을 원본 데이터 위에 그려보자. 빨간 화살표가 실제로 데이터가 퍼진 방향과 일치하는지 확인!

In [ ]:
plt.scatter(X[:, 0], X[:, 1], alpha=0.4)
for i in range(2):
    # 화살표 길이 = 고유값의 제곱근(= 표준편차)에 비례
    vec = comps[:, i] * np.sqrt(eigvals[i]) * 2
    plt.arrow(mu[0], mu[1], vec[0], vec[1],
              head_width=0.15, color='red', linewidth=2)
    plt.text(mu[0]+vec[0]*1.1, mu[1]+vec[1]*1.1,
             f'PC{i+1}', color='red', fontsize=13, fontweight='bold')
plt.axis('equal')
plt.title('원본 데이터와 주성분 축\n(빨간 화살표 길이 = 그 방향 표준편차)')
plt.xlabel('x1'); plt.ylabel('x2')
plt.show()

### 관찰 포인트
1. **PC1은 데이터가 가장 길쭉한 방향**을 정확히 가리킨다
2. **PC2는 PC1과 수직(90°)** 이며 훨씬 짧다 → 그 방향의 분산이 작다
3. 만약 1차원으로 줄인다면, 모든 점을 PC1 축으로 **수직 투영**한 값을 쓰면 된다

> 💡 **투영(Projection)이란?** PC1 축 위에 '그림자를 떨어뜨리는 것'이라고 생각하면 쉽다. 2D 점이 1D 선 위의 한 점으로 대응된다.

## 3. scikit-learn PCA와 비교

실무에서는 sklearn을 쓴다. 우리가 직접 만든 것과 결과가 같은지 확인하자.

> ⚠️ **주의**: 고유벡터는 **부호가 반대여도 같은 방향**을 나타낸다 (직선에는 방향이 두 개). 그래서 sklearn과 NumPy 결과가 부호만 다를 수 있는데, 이는 버그가 아니다.

In [ ]:
pca = PCA(n_components=2)
Z_sk = pca.fit_transform(X)
print('sklearn 설명 분산 비율:', pca.explained_variance_ratio_)
print('sklearn 고유값(분산):', pca.explained_variance_)
print('sklearn 주성분 (행 방향):\n', pca.components_)
print()
print('우리 구현과 절대값 비교:', np.allclose(np.abs(Z), np.abs(Z_sk)))

## 4. 실전: Iris 데이터셋을 2D로 시각화

### 문제 상황
붓꽃(Iris) 데이터셋은 4개의 특성(꽃받침 길이/너비, 꽃잎 길이/너비)이 있다. 4차원 데이터는 그림으로 그릴 수 없다. 어떻게 하면 데이터의 구조를 한 눈에 볼 수 있을까?

→ **PCA로 2차원으로 줄여서 그리자!**

### ⚠️ PCA 전에는 반드시 표준화!
이것은 PCA 실무에서 가장 흔히 놓치는 부분이다. 왜 표준화가 필요한가?

예를 들어 '연봉(원)'과 '나이(세)' 두 특성이 있다고 하자. 연봉은 수천만 단위, 나이는 수십 단위다. 이대로 PCA를 돌리면 **단위가 큰 연봉이 분산을 압도적으로 차지**해서 PC1은 사실상 '연봉 축'이 되어 버린다. 이는 나이 정보를 무시하는 셈이다.

**해결**: `StandardScaler`로 모든 특성을 평균 0, 분산 1로 맞춘다. 그러면 모든 특성이 공평하게 평가된다.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print('원본 데이터 shape:', X_iris.shape, '(150개 샘플, 4개 특성)')
print('클래스:', iris.target_names)

# 표준화 → PCA
X_scaled = StandardScaler().fit_transform(X_iris)
pca = PCA(n_components=2)
Z_iris = pca.fit_transform(X_scaled)

print('\n각 주성분의 설명 분산 비율:', pca.explained_variance_ratio_)
print('누적 설명 분산:', pca.explained_variance_ratio_.cumsum())
print(f'\n→ 2개 주성분만으로 전체 정보의 {pca.explained_variance_ratio_.sum()*100:.1f}%를 보존!')

In [ ]:
colors = ['red', 'green', 'blue']
for i, name in enumerate(iris.target_names):
    plt.scatter(Z_iris[y_iris==i, 0], Z_iris[y_iris==i, 1],
                c=colors[i], label=name, alpha=0.7, s=50)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Iris 데이터 — 4D를 2D로 축소')
plt.legend()
plt.show()

### 결과 해석
- **setosa(빨강)** 는 다른 두 종과 완전히 분리되어 있다 → 분류가 매우 쉬움
- **versicolor(초록)** 와 **virginica(파랑)** 는 일부 겹친다 → 두 종은 꽃 모양이 비슷함
- 단 2개의 축으로 이 정도의 분리를 볼 수 있다는 점이 놀랍다 (원본은 4차원!)

> 💡 이런 식으로 PCA는 **탐색적 데이터 분석(EDA)** 의 강력한 도구가 된다. 라벨이 없는 데이터라도 클러스터가 보이는지 확인할 수 있다.

## 5. 주성분 개수는 몇 개로 할까? — Scree Plot

### 질문
PCA로 몇 개의 주성분을 남겨야 할까? 2개? 10개? 50개? 이건 데이터마다 다르다.

### 두 가지 판단 기준
1. **누적 설명 분산 기준**: 전체 분산의 90% 또는 95%를 설명하는 최소 개수를 선택 (가장 흔함)
2. **Scree plot의 팔꿈치(elbow)**: 설명 분산이 급격히 떨어지다가 완만해지는 지점을 선택

Digits 데이터(손글씨 숫자, 8×8=64차원)에서 실습해보자.

In [ ]:
digits = load_digits()
print('원본 shape:', digits.data.shape, '(1797개 샘플, 64차원)')

X_d = StandardScaler().fit_transform(digits.data)
pca_full = PCA().fit(X_d)  # n_components 지정 안 하면 모든 주성분 계산
evr = pca_full.explained_variance_ratio_
cum = evr.cumsum()

n_95 = np.argmax(cum >= 0.95) + 1
n_90 = np.argmax(cum >= 0.90) + 1
print(f'90% 설명에 필요한 주성분 수: {n_90}개 (원본 64차원의 {n_90/64*100:.0f}%)')
print(f'95% 설명에 필요한 주성분 수: {n_95}개 (원본 64차원의 {n_95/64*100:.0f}%)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽: Scree plot
ax[0].bar(range(1, len(evr)+1), evr)
ax[0].set_title('Scree Plot — 각 주성분의 개별 기여도')
ax[0].set_xlabel('주성분 번호'); ax[0].set_ylabel('설명 분산 비율')

# 오른쪽: 누적
ax[1].plot(range(1, len(cum)+1), cum, marker='o', markersize=3)
ax[1].axhline(0.95, color='red', linestyle='--', label='95% 기준')
ax[1].axvline(n_95, color='green', linestyle='--', label=f'n={n_95}')
ax[1].set_title('누적 설명 분산')
ax[1].set_xlabel('주성분 개수'); ax[1].set_ylabel('누적 비율')
ax[1].legend()
plt.tight_layout(); plt.show()

### 해석
- 왼쪽 Scree plot을 보면 **초반 몇 개의 주성분이 전체 분산의 대부분을 차지**한다
- 뒤쪽 주성분들은 거의 기여하지 않는다 → **버려도 거의 손실 없음**
- 오른쪽 그래프에서 초록 선이 가리키는 지점이 95% 기준 최적 개수
- **결론**: 64차원을 훨씬 적은 수로 줄여도 정보의 95%를 보존할 수 있다!

## 6. PCA는 '손실 압축'이다 — 역변환으로 확인

### 핵심 개념
PCA는 **손실 압축**이다 (JPEG처럼). 주성분을 적게 쓸수록 압축률은 높지만 품질은 떨어진다.

- **압축**: `Z = X_c @ V_k` (64차원 → k차원)
- **복원**: `X_rec = Z @ V_k.T + mean` (k차원 → 64차원)

복원된 데이터는 원본과 **완전히 같지 않다**. 버린 주성분의 정보만큼 손실이 있기 때문이다. 그 손실을 눈으로 확인해보자.

In [ ]:
print(f"{'주성분 수':>8} {'누적 설명 분산':>15} {'MSE (복원 오차)':>18}")
print('-' * 50)
for n in [2, 8, 16, 32, 64]:
    pca_n = PCA(n_components=n).fit(digits.data)
    X_rec = pca_n.inverse_transform(pca_n.transform(digits.data))
    err = np.mean((digits.data - X_rec) ** 2)
    print(f'{n:>8d} {pca_n.explained_variance_ratio_.sum():>15.3f} {err:>18.3f}')

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(10, 7))
row_labels = ['원본', 'n=2', 'n=8', 'n=16', 'n=32']

for col in range(8):
    axes[0, col].imshow(digits.data[col].reshape(8, 8), cmap='gray')
    axes[0, col].axis('off')

for row, n in enumerate([2, 8, 16, 32], start=1):
    pca_n = PCA(n_components=n).fit(digits.data)
    X_rec = pca_n.inverse_transform(pca_n.transform(digits.data))
    for col in range(8):
        axes[row, col].imshow(X_rec[col].reshape(8, 8), cmap='gray')
        axes[row, col].axis('off')

for row, label in enumerate(row_labels):
    axes[row, 0].text(-5, 4, label, fontsize=12, fontweight='bold')

plt.suptitle('PCA 복원 품질 — 주성분 수가 늘수록 원본에 가까워진다', y=1.02)
plt.tight_layout(); plt.show()

### 관찰
- **n=2**: 숫자가 거의 알아볼 수 없음 — 너무 극단적 압축
- **n=8**: 대략적인 형태는 보임
- **n=16**: 숫자를 구분할 수 있을 정도
- **n=32**: 거의 원본과 구분하기 어려움 (64차원의 절반으로도 충분!)

> 💡 이게 바로 PCA가 **이미지 압축**이나 **노이즈 제거**에 쓰이는 이유다.

## 7. 주성분 해석하기 — 각 PC는 무엇을 의미하나?

PCA의 단점 중 하나는 **해석이 어렵다**는 것이다. PC1은 원본 특성들의 가중합(선형결합)이기 때문에, "PC1이 정확히 무엇을 뜻하는가"를 말하기는 쉽지 않다.

하지만 `pca.components_`를 보면 **각 원본 특성이 PC에 얼마나 기여하는지** 알 수 있다. 이를 **로딩(loading)** 이라고 부른다.

In [ ]:
pca_iris = PCA(n_components=2).fit(StandardScaler().fit_transform(X_iris))
loadings = pd.DataFrame(
    pca_iris.components_.T,
    columns=['PC1', 'PC2'],
    index=iris.feature_names
)
print('Iris PCA 로딩 (각 원본 특성의 주성분 기여도):')
print(loadings.round(3))

# 히트맵으로 시각화
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(loadings.values, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks([0, 1]); ax.set_xticklabels(['PC1', 'PC2'])
ax.set_yticks(range(4)); ax.set_yticklabels(iris.feature_names)
for i in range(4):
    for j in range(2):
        ax.text(j, i, f'{loadings.values[i,j]:.2f}', ha='center', va='center')
plt.colorbar(im)
plt.title('PCA 로딩 히트맵')
plt.tight_layout(); plt.show()

### 로딩 해석 방법
- **절대값이 큰 특성**이 해당 PC에 가장 큰 영향을 준다
- **부호**는 방향을 뜻한다 (양수 = 함께 증가, 음수 = 반대로)
- 예: PC1에서 petal length, petal width의 절대값이 크면 → **PC1은 주로 '꽃잎 크기'를 나타내는 축**

이렇게 해석하면 "PC1이 높은 샘플 = 꽃잎이 큰 샘플"이라고 이해할 수 있다.

## 8. 실습 과제: Wine 데이터셋

와인의 13가지 화학 성분(알코올, 페놀, 색상 강도 등)으로 3가지 와인 품종을 구분하는 데이터셋이다. 아래 과제를 직접 해보자.

### 과제
1. `load_wine()`으로 데이터 불러오기
2. **표준화** 후 PCA 적용
3. 2차원으로 투영하고 품종별 산점도 그리기
4. 누적 설명 분산 90% 이상이 되는 **최소 주성분 개수** 찾기
5. **보너스**: PC1과 PC2의 로딩을 확인하여 어떤 화학 성분이 품종 구분에 가장 중요한지 분석

> 💪 힌트: 위의 Iris 예제를 그대로 따라하면 된다!

In [ ]:
wine = load_wine()
print('Wine 데이터 shape:', wine.data.shape)
print('특성:', wine.feature_names)
print('클래스:', wine.target_names)

# TODO: 여기에 코드를 작성하세요


## 9. 정리 — PCA 체크리스트

### ✅ PCA를 쓸 때 기억할 것
1. **표준화 먼저!** — 특성들의 단위가 다르면 반드시 `StandardScaler` 적용
2. **선형 관계만 잡는다** — 데이터가 곡선 구조면 PCA는 한계가 있음
3. **해석 가능성 희생** — 새로운 축은 원본 특성들의 혼합이라 해석이 어려움
4. **주성분 개수 선택**: Scree plot 또는 누적 설명 분산 90~95% 기준
5. **분류/회귀의 전처리**로 쓸 때는 **학습 데이터에만 fit** → 테스트 데이터엔 transform만

### 🎯 PCA의 전형적 용도
| 용도 | 설명 |
|---|---|
| **시각화** | 고차원 데이터를 2D/3D로 축소해 탐색 |
| **노이즈 제거** | 작은 주성분을 버려서 신호만 남김 |
| **계산 속도 향상** | 특성 수를 줄여 학습 속도 개선 |
| **다중공선성 해소** | 상관된 특성을 독립 축으로 변환 |
| **이미지 압축** | 소수의 주성분으로 이미지 근사 |

### ❌ PCA의 한계와 대안
| 상황 | 대안 |
|---|---|
| 비선형 구조 | **Kernel PCA**, **t-SNE**, **UMAP** |
| 클래스 구분 최대화 원함 | **LDA**(선형판별분석) |
| 희소/음수없는 데이터 | **NMF**(비음수 행렬 분해) |
| 해석 가능성 필요 | **Factor Analysis**, **Sparse PCA** |

---
수고하셨습니다! 🎉